In [ ]:
import torch
from torch import nn
import d2l.torch as d2l

# 超参：少量训练样本，更容易看出过拟合
n_train, n_test = 20, 100
# 真实模型：4次多项式
true_w = torch.tensor([0.05, 1.0, -2.2, 1.1])
true_b = 0.08

# 构造多项式特征 x, x^2, x^3, x^4
def poly_features(x, degree):
    features = torch.ones((x.shape[0], degree))
    for i in range(1, degree):
        features[:, i] = x ** i
    return features

# 生成带高斯噪声的数据
torch.manual_seed(42)
x_train = torch.randn(n_train)
x_test = torch.randn(n_test)

# 标签由4阶多项式生成
def label_func(x):
    return poly_features(x, len(true_w)) @ true_w + true_b
train_labels = label_func(x_train) + torch.normal(0, 0.1, (n_train,))
test_labels = label_func(x_test) + torch.normal(0, 0.1, (n_test,))

In [ ]:
def evaluate_loss(net, data_iter, loss):
    metric = d2l.Accumulator(2)
    for X, y in data_iter:
        out = net(X)
        y = y.reshape(out.shape)
        l = loss(out, y)
        metric.add(l.sum(), l.numel())
    return metric[0] / metric[1]

In [ ]:
def train(train_features, test_features, train_labels, test_labels,
          lr=0.01, num_epochs=100):
    batch_size = 5
    train_iter = d2l.load_array((train_features, train_labels.reshape(-1,1)), batch_size)
    test_iter = d2l.load_array((test_features, test_labels.reshape(-1,1)), batch_size, is_train=False)

    net = nn.Sequential(nn.Linear(train_features.shape[1], 1))
    # 权重初始化
    for layer in net:
        nn.init.normal_(layer.weight, std=0.01)
        nn.init.zeros_(layer.bias)
        
    loss = nn.MSELoss()
    trainer = torch.optim.SGD(net.parameters(), lr=lr)
    
    animator = d2l.Animator(xlabel='epoch', ylabel='loss', yscale='log',
                            xlim=[1, num_epochs], legend=['train loss', 'test loss'])
    
    for epoch in range(num_epochs):
        for X, y in train_iter:
            trainer.zero_grad()
            l = loss(net(X), y)
            l.backward()
            trainer.step()
        if (epoch + 1) % 5 == 0:
            animator.add(epoch + 1,
                         (evaluate_loss(net, train_iter, loss),
                          evaluate_loss(net, test_iter, loss)))
    print('最终权重：', net[0].weight.data)

In [ ]:

degree = 4
train_features = poly_features(x_train, degree)
test_features = poly_features(x_test, degree)

train(train_features, test_features, train_labels, test_labels,
      lr=0.01, num_epochs=150)